# DetectRL DistilBERT Fast Check Notebook

This notebook mirrors the main training workflow but is reduced to a quick smoke test.
It is intended to verify that the data pipeline, model, trainer, and checkpointing all work before running the full notebook.

In [13]:
from __future__ import annotations

import json
import math
import os
import random
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DistilBertModel

from src.data.dataloader import get_dataloaders, get_unseen_loader
from src.models.distilbert_classifier import DistilBertClassifier, count_trainable_parameters, get_model_config
from src.training.trainer import Trainer

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data").exists():
    ROOT_DIR = ROOT_DIR.parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
ARTIFACT_DIR = ROOT_DIR / "artifacts" / "distilbert_detector_fastcheck"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = ROOT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}", flush=True)
print(f"Processed data: {PROCESSED_DIR}", flush=True)
print(f"Artifacts: {ARTIFACT_DIR}", flush=True)

Project root: c:\Users\Rafay\Desktop\ANN PROJECT
Processed data: c:\Users\Rafay\Desktop\ANN PROJECT\data\processed
Artifacts: c:\Users\Rafay\Desktop\ANN PROJECT\artifacts\distilbert_detector_fastcheck


In [14]:
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 2
BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
TOKENIZER_NAME = "distilbert-base-uncased"
SMOKE_LIMIT = 1000


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print(f"Device: {DEVICE}", flush=True)
print(f"Seed: {SEED}", flush=True)


Device: cuda
Seed: 42


In [15]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
label_to_id = {"human": 0, "ai": 1}
id_to_label = {v: k for k, v in label_to_id.items()}
print(tokenizer.name_or_path, flush=True)
print(label_to_id, flush=True)

c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


distilbert-base-uncased
{'human': 0, 'ai': 1}


In [18]:
import pandas as pd
df = pd.read_parquet("data/processed/train_pool.parquet")
n_per_class = max(1, SMOKE_LIMIT // 2)
df_human = df[df["label"] == 0].sample(n=min(n_per_class, len(df[df["label"] == 0])), random_state=42)
df_ai = df[df["label"] == 1].sample(n=min(n_per_class, len(df[df["label"] == 1])), random_state=42)
df_capped = pd.concat([df_human, df_ai]).sample(frac=1, random_state=42).reset_index(drop=True)
df_capped.to_parquet("data/processed/train_pool_capped.parquet", index=False)
print(f"Saved: {len(df_capped)} rows", flush=True)


Saved: 1000 rows


In [19]:
train_loader, val_loader, test_loader = get_dataloaders(
    parquet_path=PROCESSED_DIR / "train_pool_capped.parquet",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
unseen_loader = get_unseen_loader(
    parquet_path=PROCESSED_DIR / "test_unseen.parquet",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

sample_batch = next(iter(train_loader))
print({key: value.shape for key, value in sample_batch.items()}, flush=True)

Loading train_pool DataLoaders

Loaded 1,000 samples from train_pool_capped.parquet
Class distribution:
label
1    500
0    500

Using tokenizer: distilbert-base-uncased

Creating 80/10/10 train/val/test split (stratified by label)...
  Train: 800 samples (80.0%)
  Val:   100 samples (10.0%)
  Test:  100 samples (10.0%)

Creating DetectRLDataset objects...

Creating DataLoaders...

Fetching sample batch from train_loader...
  input_ids shape:    torch.Size([16, 512])
  attention_mask shape: torch.Size([16, 512])
  labels shape:       torch.Size([16])
  Batch device:       cpu

✓ DataLoaders ready!
  - train_loader: 50 batches of 16
  - val_loader:   7 batches of 16
  - test_loader:  7 batches of 16
Loading test_unseen DataLoader

Loaded 22,366 samples from test_unseen.parquet
Class distribution:
label
1    11183
0    11183
Attack type distribution:
attack_type
paraphrase_dipper_llm              13427
paraphrase_polish_llm               2259
prompt_few_shot                     2239
prom

c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Using tokenizer: distilbert-base-uncased

Creating DetectRLDataset object...
Creating DataLoader...

Fetching sample batch from unseen_loader...
  input_ids shape:    torch.Size([16, 512])
  attention_mask shape: torch.Size([16, 512])
  labels shape:       torch.Size([16])
  Batch device:       cpu

✓ Unseen DataLoader ready!
  - unseen_loader: 1398 batches of 16
{'input_ids': torch.Size([16, 512]), 'attention_mask': torch.Size([16, 512]), 'labels': torch.Size([16])}


In [21]:
def compute_metrics(logits: torch.Tensor, labels: torch.Tensor) -> dict[str, float]:
    probabilities = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
    predictions = logits.argmax(dim=-1).detach().cpu().numpy()
    targets = labels.detach().cpu().numpy()

    metrics = {
        "accuracy": accuracy_score(targets, predictions),
        "precision": precision_score(targets, predictions, zero_division=0),
        "recall": recall_score(targets, predictions, zero_division=0),
        "f1": f1_score(targets, predictions, zero_division=0),
    }
    try:
        metrics["roc_auc"] = roc_auc_score(targets, probabilities)
    except ValueError:
        metrics["roc_auc"] = float("nan")
    return metrics


def move_batch_to_device(batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {key: value.to(DEVICE) for key, value in batch.items()}


def run_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> dict[str, float]:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    all_logits = []
    all_labels = []

    for batch in loader:
        batch = move_batch_to_device(batch)
        labels = batch["labels"]

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        else:
            with torch.no_grad():
                logits = model(batch["input_ids"], batch["attention_mask"])
                loss = criterion(logits, labels)

        running_loss += loss.item() * labels.size(0)
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.detach().cpu())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)
    metrics = compute_metrics(logits, labels)
    metrics["loss"] = running_loss / len(loader.dataset)
    return metrics


def train_model(ablation_name: str) -> dict[str, Any]:
    config = get_model_config(ablation_name)
    model = DistilBertClassifier(**config).to(DEVICE)
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_f1 = -1.0
    best_state = None
    history: list[dict[str, float]] = []

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.perf_counter()
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)
        history.append({"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}})
        epoch_time = round(time.perf_counter() - epoch_start, 2)
        print(f"[{ablation_name}] epoch={epoch} train={train_metrics} val={val_metrics}", flush=True)
        print(f"[{ablation_name}] epoch={epoch} time={epoch_time}s", flush=True)

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {
                "model_state_dict": model.state_dict(),
                "config": config,
                "ablation_name": ablation_name,
                "epoch": epoch,
                "val_metrics": val_metrics,
            }

    assert best_state is not None
    model.load_state_dict(best_state["model_state_dict"])
    test_metrics = run_epoch(model, test_loader)
    unseen_metrics = run_epoch(model, unseen_loader)

    return {
        "ablation_name": ablation_name,
        "config": config,
        "history": history,
        "best_val_f1": best_val_f1,
        "test_metrics": test_metrics,
        "unseen_metrics": unseen_metrics,
        "model_state_dict": model.state_dict(),
    }


results = {}
training_times = {}
SMOKE_LIMIT = 1000

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(train_loader.dataset, list(range(min(SMOKE_LIMIT, len(train_loader.dataset))))),
    batch_size=train_loader.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=PIN_MEMORY,
)
val_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(val_loader.dataset, list(range(min(250, len(val_loader.dataset))))),
    batch_size=val_loader.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY,
)
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(test_loader.dataset, list(range(min(250, len(test_loader.dataset))))),
    batch_size=test_loader.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY,
)
unseen_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(unseen_loader.dataset, list(range(min(250, len(unseen_loader.dataset))))),
    batch_size=unseen_loader.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY,
)
for ablation_name in ["baseline1", "ablation_a", "ablation_b", "ablation_c"]:
    t_start = time.perf_counter()
    results[ablation_name] = train_model(ablation_name)
    training_times[ablation_name] = round(time.perf_counter() - t_start, 2)
    print(f"[{ablation_name}] total training time: {training_times[ablation_name]}s", flush=True)

summary_rows = []
for name, result in results.items():
    row = {"ablation_name": name, "best_val_f1": result["best_val_f1"], **{f"test_{k}": v for k, v in result["test_metrics"].items()}, **{f"unseen_{k}": v for k, v in result["unseen_metrics"].items()}} 
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[baseline1] epoch=1 train={'accuracy': 0.71875, 'precision': 0.7049180327868853, 'recall': 0.7525, 'f1': 0.727932285368803, 'roc_auc': 0.808175, 'loss': 0.5458477503061294} val={'accuracy': 0.89, 'precision': 0.8823529411764706, 'recall': 0.9, 'f1': 0.8910891089108911, 'roc_auc': 0.9696, 'loss': 0.2913566255569458}
[baseline1] epoch=1 time=35.68s
[baseline1] epoch=2 train={'accuracy': 0.875, 'precision': 0.8968253968253969, 'recall': 0.8475, 'f1': 0.87146529562982, 'roc_auc': 0.9451, 'loss': 0.3056765891611576} val={'accuracy': 0.92, 'precision': 0.8888888888888888, 'recall': 0.96, 'f1': 0.9230769230769231, 'roc_auc': 0.9828, 'loss': 0.19574712574481964}
[baseline1] epoch=2 time=34.99s
[baseline1] total training time: 76.62s


c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[ablation_a] epoch=1 train={'accuracy': 0.72125, 'precision': 0.7335092348284961, 'recall': 0.695, 'f1': 0.7137355584082157, 'roc_auc': 0.8107562499999998, 'loss': 0.5824259305000306} val={'accuracy': 0.92, 'precision': 0.92, 'recall': 0.92, 'f1': 0.92, 'roc_auc': 0.9684, 'loss': 0.32648988962173464}
[ablation_a] epoch=1 time=35.66s
[ablation_a] epoch=2 train={'accuracy': 0.895, 'precision': 0.8930348258706468, 'recall': 0.8975, 'f1': 0.8952618453865336, 'roc_auc': 0.9462937499999999, 'loss': 0.2953817383944988} val={'accuracy': 0.93, 'precision': 0.8909090909090909, 'recall': 0.98, 'f1': 0.9333333333333333, 'roc_auc': 0.9808, 'loss': 0.20166147232055665}
[ablation_a] epoch=2 time=36.03s
[ablation_a] total training time: 77.6s


c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[ablation_b] epoch=1 train={'accuracy': 0.6675, 'precision': 0.6691919191919192, 'recall': 0.6625, 'f1': 0.6658291457286433, 'roc_auc': 0.6950125000000001, 'loss': 0.6444193351268769} val={'accuracy': 0.8, 'precision': 0.875, 'recall': 0.7, 'f1': 0.7777777777777778, 'roc_auc': 0.9408000000000001, 'loss': 0.46957758903503416}
[ablation_b] epoch=1 time=21.46s
[ablation_b] epoch=2 train={'accuracy': 0.86125, 'precision': 0.8772845953002611, 'recall': 0.84, 'f1': 0.8582375478927203, 'roc_auc': 0.93100625, 'loss': 0.36205865114927294} val={'accuracy': 0.92, 'precision': 0.9038461538461539, 'recall': 0.94, 'f1': 0.9215686274509803, 'roc_auc': 0.9744, 'loss': 0.21576183080673217}
[ablation_b] epoch=2 time=21.45s
[ablation_b] total training time: 48.94s


c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[ablation_c] epoch=1 train={'accuracy': 0.695, 'precision': 0.6455223880597015, 'recall': 0.865, 'f1': 0.7393162393162394, 'roc_auc': 0.78793125, 'loss': 0.6371859085559844} val={'accuracy': 0.88, 'precision': 0.8518518518518519, 'recall': 0.92, 'f1': 0.8846153846153846, 'roc_auc': 0.956, 'loss': 0.48972578763961794}
[ablation_c] epoch=1 time=21.47s
[ablation_c] epoch=2 train={'accuracy': 0.85875, 'precision': 0.8525798525798526, 'recall': 0.8675, 'f1': 0.8599752168525403, 'roc_auc': 0.9333875, 'loss': 0.3825631168484688} val={'accuracy': 0.92, 'precision': 0.9038461538461539, 'recall': 0.94, 'f1': 0.9215686274509803, 'roc_auc': 0.976, 'loss': 0.22972527384757996}
[ablation_c] epoch=2 time=21.53s
[ablation_c] total training time: 48.85s


,ablation_name,best_val_f1,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_loss,unseen_accuracy,unseen_precision,unseen_recall,unseen_f1,unseen_roc_auc,unseen_loss
0,baseline1,0.923077,0.87,0.849057,0.90,0.873786,0.9476,0.298185,0.892,1.0,0.892,0.942918,NaN,0.283433
1,ablation_a,0.933333,0.89,0.830508,0.98,0.899083,0.9552,0.286836,0.936,1.0,0.936,0.966942,NaN,0.207026
2,ablation_b,0.921569,0.83,0.883721,0.76,0.817204,0.9392,0.327008,0.828,1.0,0.828,0.905908,NaN,0.410915
3,ablation_c,0.921569,0.87,0.893617,0.84,0.865979,0.9320,0.327198,0.856,1.0,0.856,0.922414,NaN,0.320309


In [22]:
best_ablation_name = summary_df.sort_values("best_val_f1", ascending=False).iloc[0]["ablation_name"]
best_result = results[best_ablation_name]

for name, result in results.items():
    checkpoint_path = ARTIFACT_DIR / f"{name}_best.pt"
    torch.save(result, checkpoint_path)

with open(ARTIFACT_DIR / "training_times.json", "w", encoding="utf-8") as f:
    json.dump(training_times, f, indent=2)

with open(ARTIFACT_DIR / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_ablation_name": best_ablation_name,
            "config": best_result["config"],
            "best_val_f1": float(best_result["best_val_f1"]),
        },
        f,
        indent=2,
    )

tokenizer.save_pretrained(ARTIFACT_DIR)
print(f"Best ablation: {best_ablation_name}", flush=True)
print(f"Saved checkpoint: {ARTIFACT_DIR / f'{best_ablation_name}_best.pt'}", flush=True)
print(summary_df.sort_values('best_val_f1', ascending=False).reset_index(drop=True), flush=True)

Best ablation: ablation_a
Saved checkpoint: c:\Users\Rafay\Desktop\ANN PROJECT\artifacts\distilbert_detector_fastcheck\ablation_a_best.pt
  ablation_name  best_val_f1  test_accuracy  test_precision  test_recall  \
0    ablation_a     0.933333           0.89        0.830508         0.98   
1     baseline1     0.923077           0.87        0.849057         0.90   
2    ablation_b     0.921569           0.83        0.883721         0.76   
3    ablation_c     0.921569           0.87        0.893617         0.84   

    test_f1  test_roc_auc  test_loss  unseen_accuracy  unseen_precision  \
0  0.899083        0.9552   0.286836            0.936               1.0   
1  0.873786        0.9476   0.298185            0.892               1.0   
2  0.817204        0.9392   0.327008            0.828               1.0   
3  0.865979        0.9320   0.327198            0.856               1.0   

   unseen_recall  unseen_f1  unseen_roc_auc  unseen_loss  
0          0.936   0.966942             NaN   